# Import

In [2]:
%load_ext autoreload
%autoreload 2

# Import required packages
import torch
import numpy as np
import normflows as nf
from torch.utils.data import TensorDataset, DataLoader, random_split

from matplotlib import pyplot as plt

from tqdm import tqdm

import joblib
import sklearn
import datetime

import sys
sys.path.append('../src')
import ice

from scipy.io import loadmat
torch.multiprocessing.set_start_method('spawn', force=True)

In [3]:
# load simulation data
Em = loadmat("../data/NGrIS/ensemble_data_pixelated/enthalpy_avg_grid.mat")['enthalpy_avg_grid']
pmp = loadmat("../data/NGrIS/ensemble_data_pixelated/pmp_grid.mat")['pmp_grid']
boundary = loadmat("../data/NGrIS/ensemble_data_pixelated/boundary_grid.mat")['boundary_grid']
radar_mask = loadmat("../data/NGrIS/ensemble_data_pixelated/radar_mask.mat")['radar_mask']
radar_mask = (radar_mask == 1)

Em = np.array(Em)
pmp = pmp[:, :, np.newaxis]
pmp = np.tile(pmp, (1, 1, Em.shape[2]))

# convert Em to attenu rate
Tm = ice.enthalpy_to_temperature(Em, Tpmp = pmp, istorch = False)

In [ ]:
# load holocene and LGP ice fraction data
Hol_depth = loadmat('../data/NGrIS/ensemble_data_pixelated/holocene_depth_grid.mat')['holocene_depth_grid']
Hol_depth_uncert = loadmat('../data/NGrIS/ensemble_data_pixelated/holocene_depth_uncert_grid.mat')['holocene_depth_uncert_grid']
LGP_depth = loadmat('../data/NGrIS/ensemble_data_pixelated/lgp_depth_grid.mat')['lgp_depth_grid']
LGP_depth_uncert = loadmat('../data/NGrIS/ensemble_data_pixelated/lgp_depth_uncert_grid.mat')['lgp_depth_uncert_grid']
H = loadmat('../data/NGrIS/ensemble_data_pixelated/H_grid.mat')['H_grid']

# assuming report uncertainty is 1 standard deviation, generate Gaussian samples
#Hol_depth = np.random.multivariate_normal(mean = Hol_depth.flatten(), cov = np.diag(Hol_depth_uncert.flatten()))

Hol_valid_mask = (radar_mask) & (~np.isnan(Hol_depth) & (~np.isnan(Hol_depth_uncert)))
Hol_Gaussian = torch.distributions.Normal(loc=torch.tensor(Hol_depth[Hol_valid_mask].flatten()), scale=torch.tensor(Hol_depth_uncert[Hol_valid_mask].flatten()))
Hol_samples = Hol_Gaussian.sample((1000,))

LGP_valid_mask = (radar_mask) & (~np.isnan(LGP_depth) & (~np.isnan(LGP_depth_uncert)))
LGP_Gaussian = torch.distributions.Normal(loc=torch.tensor(LGP_depth[LGP_valid_mask].flatten()), scale=torch.tensor(LGP_depth_uncert[LGP_valid_mask].flatten()))
LGP_samples = LGP_Gaussian.sample((1000,))